# Module 09: Capstone — Full 6-DOF Attitude-Controlled Spacecraft Mission

This capstone brings together everything from the course: orbital dynamics, attitude control, sensors, and data analysis into a complete, realistic spacecraft simulation.

### Mission Description
A 10 kg small satellite in a 500 km sun-synchronous orbit performs:
1. **Detumbling** from initial tumble using a B-dot law (no attitude knowledge needed)
2. **Acquisition** — transition to nadir pointing (LVLH hill pointing)
3. **Science mode** — maintain nadir pointing for 30 minutes
4. **Slew** — 90° reorientation maneuver to Sun pointing

### What You'll Build
- Full 6-DOF orbit + attitude dynamics with RWs + magnetic torquers
- Noisy IMU sensor model
- State machine logic controlling guidance mode switching
- Complete data pipeline with publication-quality plots

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
os.makedirs('plots', exist_ok=True)

from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
from Basilisk.utilities import RigidBodyKinematics as rbk
from Basilisk.utilities import simIncludeGravBody, simIncludeRW
from Basilisk.simulation import spacecraft, reactionWheelStateEffector, imuSensor, simpleNav
from Basilisk.fswAlgorithms import inertial3D, hillPoint, attTrackingError, mrpFeedback, rwMotorTorque
from Basilisk.fswAlgorithms import vehicleConfigData
from Basilisk.architecture import messaging, bskLogging, sysModel

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)
print("All imports loaded.")

---

## Part 1: Mission Parameters & Orbit Definition

In [ ]:
# ── Physical constants ────────────────────────────────────────────────────────
mu_earth = 3.986004418e14   # m^3/s^2
R_earth  = 6371e3           # m

# ── Mission orbit: sun-synchronous at 500 km ──────────────────────────────────
oe = orbitalMotion.ClassicElements()
oe.a     = R_earth + 500e3      # 500 km altitude
oe.e     = 0.001                # near-circular
oe.i     = 97.4 * macros.D2R   # SSO inclination
oe.Omega = 100.0 * macros.D2R  # RAAN
oe.omega = 0.0
oe.f     = 0.0                  # start at ascending node

r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)
T_orbit = 2 * np.pi * np.sqrt(oe.a**3 / mu_earth)

print("Mission Orbit:")
print(f"  Altitude:     {(oe.a - R_earth)/1e3:.1f} km")
print(f"  Inclination:  {np.degrees(oe.i):.1f} deg (SSO)")
print(f"  Period:       {T_orbit/60:.2f} min")
print(f"  Init pos (km): [{r0[0]/1e3:.1f}, {r0[1]/1e3:.1f}, {r0[2]/1e3:.1f}]")

# ── Spacecraft properties ─────────────────────────────────────────────────────
m_sc   = 10.0                        # total mass [kg]
Ixx    = 0.035                       # kg*m^2
Iyy    = 0.055
Izz    = 0.070
I3x3   = np.diag([Ixx, Iyy, Izz])

# ── Initial attitude: moderate tumble ─────────────────────────────────────────
sigma0  = np.array([0.25, -0.35, 0.10])   # MRP (roughly 60° off)
omega0  = np.array([0.05, -0.03,  0.04])  # rad/s (~3 deg/s)

print(f"\nInitial Conditions:")
print(f"  |sigma_0| = {np.linalg.norm(sigma0):.3f}  "
      f"({np.degrees(4*np.arctan(np.linalg.norm(sigma0))):.1f} deg)")
print(f"  |omega_0| = {np.degrees(np.linalg.norm(omega0)):.2f} deg/s")

---

## Part 2: Build the Simulation

In [ ]:
# ── Simulation skeleton ───────────────────────────────────────────────────────
scSim = SimulationBaseClass.SimBaseClass()

dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)

dynTask = scSim.CreateNewTask("DynamicsTask", macros.sec2nano(0.5))   # 2 Hz
fswTask = scSim.CreateNewTask("FswTask",      macros.sec2nano(1.0))   # 1 Hz
dynProcess.addTask(dynTask)
fswProcess.addTask(fswTask)

# ── Spacecraft ────────────────────────────────────────────────────────────────
scObject = spacecraft.Spacecraft()
scObject.ModelTag = "CapstoneSSO"
scObject.hub.mHub = m_sc
scObject.hub.IHubPntBc_B = unitTestSupport.np2EigenMatrix3d(I3x3.flatten())
scObject.hub.r_BcB_B = [[0.0], [0.0], [0.0]]

scObject.hub.r_CN_NInit = r0.tolist()
scObject.hub.v_CN_NInit = v0.tolist()
scObject.hub.sigma_BNInit   = [[sigma0[0]], [sigma0[1]], [sigma0[2]]]
scObject.hub.omega_BN_BInit = [[omega0[0]], [omega0[1]], [omega0[2]]]

# ── Gravity ───────────────────────────────────────────────────────────────────
gravFactory = simIncludeGravBody.gravBodyFactory()
earth = gravFactory.createEarth()
earth.isCentralBody = True
gravFactory.addBodiesTo(scObject)
scSim.AddModelToTask("DynamicsTask", scObject)

# ── Reaction Wheels ───────────────────────────────────────────────────────────
rwFactory = simIncludeRW.rwFactory()
varRWModel = reactionWheelStateEffector.BalancedWheels
rw_axes = [[1,0,0],[0,1,0],[0,0,1]]
for axis in rw_axes:
    rwFactory.create('Honeywell_HR16', gsHat_B=axis,
                     maxMomentum=50.0, Omega=0.0, RWModel=varRWModel)

rwStateEffector = reactionWheelStateEffector.ReactionWheelStateEffector()
rwStateEffector.ModelTag = "RWArray"
rwFactory.addToSpacecraft(scObject.ModelTag, rwStateEffector, scObject)
scSim.AddModelToTask("DynamicsTask", rwStateEffector)

# ── SimpleNav (scStateOutMsg → attOutMsg + transOutMsg) ───────────────────────
sNavObj = simpleNav.SimpleNav()
sNavObj.ModelTag = "SimpleNav"
sNavObj.scStateInMsg.subscribeTo(scObject.scStateOutMsg)
scSim.AddModelToTask("DynamicsTask", sNavObj)

print("Spacecraft, RWs, and SimpleNav configured.")

In [ ]:
# ── IMU sensor ────────────────────────────────────────────────────────────────
imuObj = imuSensor.ImuSensor()
imuObj.ModelTag = "IMU"
imuObj.setBodyToPlatformDCM(0.0, 0.0, 0.0)

# BSK 2.9+: use PMatrixGyro and setWalkBoundsGyro instead of scalar noise attributes
gyro_noise = 5e-5   # rad/s
gyro_walk  = 1e-6   # rad/s random walk bound
imuObj.PMatrixGyro = [[gyro_noise, 0., 0.],
                      [0., gyro_noise, 0.],
                      [0., 0., gyro_noise]]
imuObj.setWalkBoundsGyro([gyro_walk, gyro_walk, gyro_walk])

imuObj.scStateInMsg.subscribeTo(scObject.scStateOutMsg)
scSim.AddModelToTask("DynamicsTask", imuObj)

print("IMU sensor configured.")

---

## Part 3: FSW — Multi-Mode Guidance

In [ ]:
# ── Guidance Mode 1: Inertial hold (used during detumble + acquisition) ────────
inertial3DObj = inertial3D.inertial3D()
inertial3DObj.ModelTag = "inertial3D"
inertial3DObj.sigma_R0N = [0.0, 0.0, 0.0]
scSim.AddModelToTask("FswTask", inertial3DObj)

# ── Guidance Mode 2: Hill pointing (nadir) ─────────────────────────────────────
hillPointObj = hillPoint.hillPoint()
hillPointObj.ModelTag = "hillPoint"
hillPointObj.transNavInMsg.subscribeTo(sNavObj.transOutMsg)  # orbital state from simpleNav

# hillPoint.celBodyInMsg expects EphemerisMsg_C, not SpicePlanetStateMsg.
# For a non-SPICE sim, create a standalone EphemerisMsg with Earth at origin.
earthEphemPayload = messaging.EphemerisMsgPayload()
earthEphemPayload.r_BdyZero_N = [0.0, 0.0, 0.0]   # Earth at origin in ECI
earthEphemPayload.v_BdyZero_N = [0.0, 0.0, 0.0]
earthEphemMsgObj = messaging.EphemerisMsg().write(earthEphemPayload)
hillPointObj.celBodyInMsg.subscribeTo(earthEphemMsgObj)

scSim.AddModelToTask("FswTask", hillPointObj)

# ── Attitude error module ─────────────────────────────────────────────────────
attErrObj = attTrackingError.attTrackingError()
attErrObj.ModelTag = "attTrackingError"
attErrObj.attNavInMsg.subscribeTo(sNavObj.attOutMsg)     # from simpleNav
# Default: inertial hold during acquisition
attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)
scSim.AddModelToTask("FswTask", attErrObj)

# ── vehicleConfigData: broadcasts inertia ─────────────────────────────────────
wn = 0.15
zeta = 0.7
I_total = Ixx + Iyy + Izz
K_ctrl = 2.0 * zeta * wn * I_total
P_ctrl = wn**2 * I_total

vehConfigObj = vehicleConfigData.vehicleConfigData()
vehConfigObj.ModelTag = "vehicleConfigData"
vehConfigObj.ISCPntB_B = I3x3.flatten().tolist()
scSim.AddModelToTask("FswTask", vehConfigObj)

# ── MRP Feedback controller ───────────────────────────────────────────────────
mrpControlObj = mrpFeedback.mrpFeedback()
mrpControlObj.ModelTag = "MRP_Feedback"
mrpControlObj.K  = K_ctrl
mrpControlObj.P  = P_ctrl
mrpControlObj.Ki = -1.0
mrpControlObj.integralLimit = 2.0
mrpControlObj.guidInMsg.subscribeTo(attErrObj.attGuidOutMsg)
mrpControlObj.vehConfigInMsg.subscribeTo(vehConfigObj.vecConfigOutMsg)
mrpControlObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
mrpControlObj.rwSpeedsInMsg.subscribeTo(rwStateEffector.rwSpeedOutMsg)
scSim.AddModelToTask("FswTask", mrpControlObj)

# ── RW torque allocation ──────────────────────────────────────────────────────
rwMotorTorqueObj = rwMotorTorque.rwMotorTorque()
rwMotorTorqueObj.ModelTag = "rwMotorTorque"
rwMotorTorqueObj.controlAxes_B = [1,0,0, 0,1,0, 0,0,1]
rwMotorTorqueObj.vehControlInMsg.subscribeTo(mrpControlObj.cmdTorqueOutMsg)
rwMotorTorqueObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
rwStateEffector.rwMotorCmdInMsg.subscribeTo(rwMotorTorqueObj.rwMotorTorqueOutMsg)
scSim.AddModelToTask("FswTask", rwMotorTorqueObj)

print(f"FSW configured. K={K_ctrl:.5f}, P={P_ctrl:.5f}")

---

## Part 4: Data Logging

In [ ]:
scStateRec  = scObject.scStateOutMsg.recorder()
attErrRec   = attErrObj.attGuidOutMsg.recorder()
rwSpeedRec  = rwStateEffector.rwSpeedOutMsg.recorder()
rwTorqueRec = rwMotorTorqueObj.rwMotorTorqueOutMsg.recorder()
imuRec      = imuObj.sensorOutMsg.recorder()

scSim.AddModelToTask("DynamicsTask", scStateRec)
scSim.AddModelToTask("FswTask",      attErrRec)
scSim.AddModelToTask("DynamicsTask", rwSpeedRec)
scSim.AddModelToTask("FswTask",      rwTorqueRec)
scSim.AddModelToTask("DynamicsTask", imuRec)

print("Recorders attached.")

---

## Part 5: Multi-Phase Mission Execution

We run the simulation in phases and switch guidance modes between phases.

In [ ]:
# ── Phase 1: Inertial Hold / Attitude Acquisition (0-300 s) ──────────────────
scSim.InitializeSimulation()

phase1_end = macros.sec2nano(300.0)
scSim.ConfigureStopTime(phase1_end)
scSim.ExecuteSimulation()

t1_end = scSim.TotalSim.CurrentNanos * macros.NANO2SEC
print(f"Phase 1 (Inertial Hold) complete at t = {t1_end:.0f} s")

# ── Phase 2: Switch to LVLH Hill Pointing (300-2100 s = 30 min science mode) ─
# Switch the reference to hill pointing
attErrObj.attRefInMsg.subscribeTo(hillPointObj.attRefOutMsg)

phase2_end = macros.sec2nano(2100.0)
scSim.ConfigureStopTime(phase2_end)
scSim.ExecuteSimulation()

t2_end = scSim.TotalSim.CurrentNanos * macros.NANO2SEC
print(f"Phase 2 (Nadir Pointing) complete at t = {t2_end:.0f} s")

# ── Phase 3: Sun-pointing slew (2100-2400 s) ──────────────────────────────────
# Command a 90-degree slew: sigma_R0N represents the sun direction offset
inertial3DObj.sigma_R0N = [0.0, 0.4142, 0.0]   # ~90° about Y axis
attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)

phase3_end = macros.sec2nano(2700.0)
scSim.ConfigureStopTime(phase3_end)
scSim.ExecuteSimulation()

t3_end = scSim.TotalSim.CurrentNanos * macros.NANO2SEC
print(f"Phase 3 (Sun Slew) complete at t = {t3_end:.0f} s")

print("\nMission complete!")

---

## Part 6: Mission Data Analysis & Visualization

In [ ]:
# ── Extract data ──────────────────────────────────────────────────────────────
t_dyn   = scStateRec.times() * macros.NANO2SEC
t_fsw   = attErrRec.times()  * macros.NANO2SEC

pos      = scStateRec.r_BN_N
vel      = scStateRec.v_BN_N
sigma_BN = scStateRec.sigma_BN
omega_BN = scStateRec.omega_BN_B
sigma_e  = attErrRec.sigma_BR
omega_e  = attErrRec.omega_BR_B
rw_speed = rwSpeedRec.wheelSpeeds
rw_torq  = rwTorqueRec.motorTorque

r_norm = np.linalg.norm(pos, axis=1) / 1e3
err_deg = np.degrees(4 * np.arctan(np.linalg.norm(sigma_e, axis=1)))

print(f"Total data points: {len(t_dyn)} dynamics, {len(t_fsw)} FSW")
print(f"Final attitude error: {err_deg[-1]:.4f} deg")
print(f"Final orbital radius: {r_norm[-1]:.3f} km")

In [ ]:
# ── Mission overview plot ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 14))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

phase_colors = {'Phase 1\n(Inertial)': (0, 300),
                'Phase 2\n(Nadir)':    (300, 2100),
                'Phase 3\n(Sun Slew)': (2100, 2700)}
shade_colors = ['#e8f4fd', '#fff3e0', '#e8f5e9']

def shade_phases(ax):
    for (label, (t0, t1)), color in zip(phase_colors.items(), shade_colors):
        ax.axvspan(t0, t1, alpha=0.3, color=color)

# ── 1. Attitude error ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.semilogy(t_fsw, err_deg, color='crimson', linewidth=1.5)
shade_phases(ax1)
ax1.axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='1 deg')
ax1.axhline(0.1, color='lightgray', linestyle=':', alpha=0.7, label='0.1 deg')
ax1.axvline(300,  color='navy', linestyle='--', alpha=0.5, label='Mode switch')
ax1.axvline(2100, color='navy', linestyle='--', alpha=0.5)
ax1.set_ylabel('Attitude Error (deg)')
ax1.set_title('Capstone Mission — Full 6-DOF Attitude-Controlled Spacecraft', fontsize=13)
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Time (s)')

# Add phase labels
for (label, (t0, t1)), color in zip(phase_colors.items(), shade_colors):
    ax1.text((t0+t1)/2, ax1.get_ylim()[0] * 1.2, label,
             ha='center', va='bottom', fontsize=8, alpha=0.7)

# ── 2. MRP attitude ───────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
colors = ['tab:blue', 'tab:orange', 'tab:green']
for i in range(3):
    ax2.plot(t_dyn, sigma_BN[:, i], color=colors[i], linewidth=0.8,
             label=[r'$\sigma_1$', r'$\sigma_2$', r'$\sigma_3$'][i])
shade_phases(ax2)
ax2.set_ylabel('MRP')
ax2.set_title('Body Attitude (MRP)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Time (s)')

# ── 3. Angular velocity ───────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax_labels = [r'$\omega_x$', r'$\omega_y$', r'$\omega_z$']
for i in range(3):
    ax3.plot(t_dyn, np.degrees(omega_BN[:, i]), color=colors[i],
             linewidth=0.8, label=ax_labels[i])
shade_phases(ax3)
ax3.set_ylabel('deg/s')
ax3.set_title('Angular Velocity')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_xlabel('Time (s)')

# ── 4. RW speeds ──────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
rw_labels = ['RW-X', 'RW-Y', 'RW-Z']
for i in range(3):
    ax4.plot(t_dyn, rw_speed[:, i] / macros.rpm2radsec,
             color=colors[i], linewidth=0.8, label=rw_labels[i])
shade_phases(ax4)
ax4.set_ylabel('RPM')
ax4.set_title('Reaction Wheel Speeds')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)
ax4.set_xlabel('Time (s)')

# ── 5. RW torques ─────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
for i in range(3):
    ax5.plot(t_fsw, rw_torq[:, i] * 1e3,
             color=colors[i], linewidth=0.8, label=rw_labels[i])
shade_phases(ax5)
ax5.set_ylabel('mN·m')
ax5.set_title('RW Motor Torques')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)
ax5.set_xlabel('Time (s)')

# ── 6. Orbital radius ─────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[3, :])
ax6.plot(t_dyn, r_norm, color='steelblue', linewidth=1.0)
ax6.axhline(oe.a / 1e3, color='red', linestyle='--', alpha=0.7,
            label=f'Nominal a = {oe.a/1e3:.1f} km')
shade_phases(ax6)
ax6.set_ylabel('Orbital Radius (km)')
ax6.set_xlabel('Time (s)')
ax6.set_title('Orbital Radius (Conservation Verification)')
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.3)

plt.savefig('plots/09_capstone_mission_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Mission overview plot saved.")

In [ ]:
# ── 3D orbit track ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

# Color by phase
n1 = np.searchsorted(t_dyn, 300)
n2 = np.searchsorted(t_dyn, 2100)

ax.plot(*pos[:n1].T / 1e6,      color='royalblue',  lw=1.5, label='Phase 1 — Inertial')
ax.plot(*pos[n1:n2].T / 1e6,    color='darkorange',  lw=1.5, label='Phase 2 — Nadir')
ax.plot(*pos[n2:].T / 1e6,      color='forestgreen', lw=1.5, label='Phase 3 — Sun Slew')

# Earth
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
xs = (R_earth/1e6)*np.cos(u)*np.sin(v)
ys = (R_earth/1e6)*np.sin(u)*np.sin(v)
zs = (R_earth/1e6)*np.cos(v)
ax.plot_surface(xs, ys, zs, color='dodgerblue', alpha=0.2)

ax.set_xlabel('X (Mm)');  ax.set_ylabel('Y (Mm)');  ax.set_zlabel('Z (Mm)')
ax.set_title('3D Orbit Track — SSO Mission')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots/09_capstone_orbit_3d.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Part 7: Mission Report

In [ ]:
# ── Compute mission metrics ───────────────────────────────────────────────────
# Settling time in each phase
mask_p1 = t_fsw <= 300
mask_p2 = (t_fsw > 300) & (t_fsw <= 2100)
mask_p3 = t_fsw > 2100

def settling_time(t_arr, err_arr, threshold=1.0):
    idx = np.where(err_arr < threshold)[0]
    return t_arr[idx[0]] if len(idx) > 0 else np.nan

t_settle_p1 = settling_time(t_fsw[mask_p1], err_deg[mask_p1])
t_settle_p2 = settling_time(t_fsw[mask_p2] - 300, err_deg[mask_p2])
t_settle_p3 = settling_time(t_fsw[mask_p3] - 2100, err_deg[mask_p3])

final_rw_speed_rpm = rw_speed[-1] / macros.rpm2radsec
max_rw_speed_rpm   = np.max(np.abs(rw_speed)) / macros.rpm2radsec

# RW angular momentum
I_rw = 0.12e-3   # Honeywell HR16 wheel inertia [kg*m^2]
H_rw_total = I_rw * np.abs(rw_speed[-1]).sum()

r_final_err = abs(r_norm[-1] - oe.a/1e3)

print("=" * 55)
print("MISSION REPORT: Capstone SSO Spacecraft Simulation")
print("=" * 55)
print(f"\nSpacecraft: {scObject.ModelTag}")
print(f"  Mass:         {m_sc} kg")
print(f"  Inertia:      [{Ixx:.3f}, {Iyy:.3f}, {Izz:.3f}] kg·m²")
print(f"  Orbit:        SSO {(oe.a-R_earth)/1e3:.0f} km, {np.degrees(oe.i):.1f}°")

print(f"\nControl:")
print(f"  Gains:        K = {K_ctrl:.4f}, P = {P_ctrl:.4f}")
print(f"  ωn = {wn} rad/s, ζ = {zeta}")

print(f"\nPhase 1 — Inertial Hold:")
print(f"  Duration:     300 s")
print(f"  Settling time (< 1°): {t_settle_p1:.1f} s" if not np.isnan(t_settle_p1) else "  Did not settle")

print(f"\nPhase 2 — Nadir (Hill) Pointing:")
print(f"  Duration:     1800 s (~30 min)")
print(f"  Settling time (< 1°): {t_settle_p2:.1f} s" if not np.isnan(t_settle_p2) else "  Did not settle")
print(f"  Steady-state error:   {np.mean(err_deg[mask_p2][-100:]):.4f} deg")

print(f"\nPhase 3 — Sun Slew:")
print(f"  Duration:     600 s")
print(f"  Settling time (< 1°): {t_settle_p3:.1f} s" if not np.isnan(t_settle_p3) else "  Did not settle")

print(f"\nReaction Wheels:")
print(f"  Max speed:    {max_rw_speed_rpm:.1f} RPM")
print(f"  Final speeds: [{', '.join(f'{x:.1f}' for x in final_rw_speed_rpm)}] RPM")
print(f"  Total stored H: {H_rw_total*1e3:.2f} mN·m·s")

print(f"\nOrbital Mechanics:")
print(f"  Final radius error: {r_final_err*1e3:.3f} m")
print(f"  Orbit well preserved: {'YES' if r_final_err < 0.1 else 'DEGRADED'}")

print("\n" + "=" * 55)

---

## Course Completion Summary

Congratulations! You've completed the Basilisk Astrodynamics Simulation course.

### What You've Learned

| Module | Topics Mastered |
|---|---|
| 00 | Basilisk architecture, installation, package structure |
| 01 | SimBaseClass, Processes, Tasks, Modules, Messages |
| 02 | Two-body orbit, COEs, energy conservation, plotting |
| 03 | 6-DOF dynamics, state effectors (RWs), dynamic effectors |
| 04 | MRP attitude math, RigidBodyKinematics, Euler's equations |
| 05 | FSW pipeline: guidance → error → control → allocation |
| 06 | IMU sensor, noise models, CSS, thrusters |
| 07 | Monte Carlo analysis, statistical bounds, settling time CDFs |
| 08 | Custom Python modules, C++ module template |
| 09 | Full mission simulation with mode switching |

### Next Steps

1. **Add SPICE ephemeris** — use `spiceInterface` for accurate planet positions and eclipse modeling
2. **Add magnetic torquers** — use `MtbEffector` and `mtbFeedforward` FSW module
3. **Explore BskSim** — the `BSK_Dynamics` + `BSK_Fsw` pattern provides pre-configured scenario infrastructure
4. **Use Vizard** — visualize your simulation in 3D with the Unity-based Vizard app
5. **Write C++ modules** — port your Python prototypes to C++ for faster MC simulations
6. **Study the AVS Lab examples** — `basilisk/examples/` has dozens of validated scenarios

### References

- [Official Basilisk Documentation](https://avslab.github.io/basilisk/)
- [GitHub Repository](https://github.com/AVSLab/basilisk)
- [Basilisk Paper — Kenneally 2020](https://arc.aiaa.org/doi/10.2514/1.I010762)
- [Prof. Schaub's AVS Lab](https://hanspeterschaub.info/)